# 04 · Routing: min-distance vs min-CO₂

**Goal:** solve the same capacitated vehicle routing problem (CVRP) twice, once minimising distance and once minimising CO₂, and score both with the same evaluator.

### Load data and compute climbs
Loads stops, distances and congestion-corrected durations. The climb on arc $i \to j$ uses only the elevations at the two endpoints:

$$\text{climb}_{ij} = \max(h_j - h_i,\ 0)$$

Hills *between* stops are invisible, so any gradient effect found is a lower bound. `TERRAIN_SCALE` multiplies all elevations to simulate a hillier city (1.0 = real terrain).

In [1]:
import itertools
import os

import numpy as np
import pandas as pd
from ortools.constraint_solver import pywrapcp, routing_enums_pb2

import emissions   # the CO2 model from Notebook 3

CITY = os.environ.get("EMROUTE_CITY", "dhaka")
DATA_DIR = "../data"

stops = pd.read_csv(f"{DATA_DIR}/stops_{CITY}.csv")
distance_m = pd.read_csv(f"{DATA_DIR}/distance_m_{CITY}.csv").values
duration_s = pd.read_csv(f"{DATA_DIR}/duration_s_{CITY}.csv").values

n = len(stops)
demands = stops["demand_kg"].values

NUM_VEHICLES = 3
CAPACITY_KG = 600
DEPOT = 0

# 1.0 = real terrain. Larger values exaggerate relief for a "what if it were hillier" test.
TERRAIN_SCALE = 1.0
elevation = stops["elevation_m"].values * TERRAIN_SCALE

# climb_m[i, j] = metres gained driving from i to j (0 if downhill)
climb_m = np.maximum(elevation[None, :] - elevation[:, None], 0)

print(f"{n} locations, {demands.sum():.0f} kg demand, "
      f"{NUM_VEHICLES} x {CAPACITY_KG} kg capacity")
print(f"Max climb on any arc: {climb_m.max():.0f} m")

21 locations, 1085 kg demand, 3 x 600 kg capacity
Max climb on any arc: 19 m


### Evaluator and payload tracking
`arcs_with_payload` walks a route and tracks the weight on board:
- The van leaves the depot carrying the sum of all its stops' demands.
- On arc $i \to j$ it still carries $j$'s parcel; after arriving, $\text{payload} \leftarrow \text{payload} - \text{demand}_j$.

`evaluate(routes)` is the **single referee** for every solution: it adds up distance and true CO₂ over all arcs.

`payload_leaving_each_node` records the payload as the van departs each node. It is used later to build the CO₂ cost matrix.

`print_routes` lists each van's load and stops by name.

In [2]:
def arcs_with_payload(route, demand):
    """Yield (i, j, payload) for each arc i -> j of a route.

    payload = weight on board while driving i -> j. The van leaves the depot
    with every parcel for the route and drops j's parcel on arrival at j.
    """
    payload = sum(demand[node] for node in route)
    for i, j in zip(route, route[1:]):
        yield i, j, payload
        payload -= demand[j]


def evaluate(routes):
    """Total distance (km) and true CO2 (kg) of a set of routes."""
    total_distance_m = 0.0
    total_co2 = 0.0
    for route in routes:
        for i, j, payload in arcs_with_payload(route, demands):
            total_distance_m += distance_m[i][j]
            total_co2 += emissions.co2_kg(distance_m[i][j], duration_s[i][j], payload, climb_m[i][j])
    return total_distance_m / 1000.0, total_co2


def payload_leaving_each_node(routes):
    """Payload on board as the van departs each node, as a list indexed by node."""
    payloads = [0] * n
    for route in routes:
        for i, _, payload in arcs_with_payload(route, demands):
            payloads[i] = payload
    return payloads


def print_routes(routes):
    for v, route in enumerate(routes, 1):
        load = sum(demands[i] for i in route)
        names = " -> ".join(stops.loc[i, "address"].split(",")[0] for i in route)
        print(f"  Van {v} ({load:.0f} kg): {names}")

### Generic CVRP solver (OR-Tools)
The objective is whatever cost matrix is passed in, so the same function can minimise distance or CO₂.
- **Arc cost:** `cost_matrix[i][j]`. It depends only on (from, to), with no knowledge of how loaded the van is.
- **Capacity:** each stop adds its demand to a running total that must stay ≤ 600 kg per van.
- **Search:** build a greedy first solution (*path cheapest arc*), then improve it with *guided local search* for `time_limit_s` seconds.

Routes are read out as node lists `[0, …, 0]`; unused vans (depot → depot) are dropped.

In [3]:
def solve_cvrp(cost_matrix, time_limit_s=10):
    """Solve the capacitated VRP for any integer cost matrix. Returns a list of routes."""
    # OR-Tools has its own internal indices; manager.IndexToNode() maps them to our nodes.
    manager = pywrapcp.RoutingIndexManager(n, NUM_VEHICLES, DEPOT)
    routing = pywrapcp.RoutingModel(manager)

    # Arc cost depends only on (from, to): OR-Tools cannot see the van's current load.
    def cost_callback(from_index, to_index):
        return cost_matrix[manager.IndexToNode(from_index)][manager.IndexToNode(to_index)]

    routing.SetArcCostEvaluatorOfAllVehicles(routing.RegisterTransitCallback(cost_callback))

    # Capacity: the demand served on one route must fit in one van.
    def demand_callback(from_index):
        return int(demands[manager.IndexToNode(from_index)])

    routing.AddDimensionWithVehicleCapacity(
        routing.RegisterUnaryTransitCallback(demand_callback),
        0,                              # no slack
        [CAPACITY_KG] * NUM_VEHICLES,   # capacity of each van
        True,                           # every van starts empty
        "Capacity",
    )

    # Greedy first solution, then improve it with guided local search.
    params = pywrapcp.DefaultRoutingSearchParameters()
    params.first_solution_strategy = routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC
    params.local_search_metaheuristic = routing_enums_pb2.LocalSearchMetaheuristic.GUIDED_LOCAL_SEARCH
    params.time_limit.FromSeconds(time_limit_s)

    solution = routing.SolveWithParameters(params)
    if solution is None:
        raise RuntimeError("No solution found -- check capacity vs demand.")

    routes = []
    for vehicle in range(NUM_VEHICLES):
        index = routing.Start(vehicle)
        route = []
        while not routing.IsEnd(index):
            route.append(manager.IndexToNode(index))
            index = solution.Value(routing.NextVar(index))
        route.append(manager.IndexToNode(index))   # back at the depot

        if len(route) > 2:                         # skip vans that never left the depot
            routes.append(route)

    return routes

### Solution A: minimise distance
OR-Tools needs integer costs, so distances are rounded to whole metres. The resulting routes are then scored for both distance and CO₂ with `evaluate`.

In [4]:
distance_cost = np.round(distance_m).astype(int)   # OR-Tools needs integers

routes_distance = solve_cvrp(distance_cost)
dist_A, co2_A = evaluate(routes_distance)

print(f"SOLUTION A -- minimise distance")
print(f"  vehicles used: {len(routes_distance)}")
print(f"  distance: {dist_A:.2f} km")
print(f"  CO2:      {co2_A:.3f} kg\n")
print_routes(routes_distance)

SOLUTION A -- minimise distance
  vehicles used: 2
  distance: 76.86 km
  CO2:      43.219 kg

  Van 1 (499 kg): Tejgaon Industrial Area -> Rampura -> Badda -> Bashundhara Residential Area -> Uttara Sector 7 -> Mirpur 10 -> Banani -> Gulshan 2 -> Gulshan 1 -> Mohakhali -> Tejgaon Industrial Area
  Van 2 (586 kg): Tejgaon Industrial Area -> Malibagh -> Paltan -> Motijheel -> Shahbagh -> Azimpur -> New Market -> Dhanmondi 27 -> Lalmatia -> Mohammadpur -> Farmgate -> Kawran Bazar -> Tejgaon Industrial Area


### Why CO₂ can't be passed in directly

You want arc cost = co2_kg(distance, duration, PAYLOAD, climb).

But OR-Tools gives you:      cost_callback(from_index, to_index)
                                                  ^ no payload anywhere

And payload on arc (i -> j) depends on which stops were served BEFORE i,
which depends on the route, which is what the solver is still deciding.

    payload needs the route  ->  route needs the cost  ->  cost needs the payload

That is a genuine circular dependency, not a missing API. There is no flag
to switch on.

## Solution B: minimise CO₂ by iterative re-costing

The circular dependency is broken by guessing payloads and then refining the guess:
1. Start from the payloads implied by Solution A.
2. Build a CO₂ cost matrix $c_{ij} = \text{co2\_kg}(d_{ij},\ t_{ij},\ p_i,\ \text{climb}_{ij})$, where $p_i$ is the estimated payload leaving node $i$. Multiply by $10^6$ (kg → mg) and round to integers for OR-Tools.
3. Solve, measure the payloads the new routes actually imply, and repeat.
4. Stop when a route set repeats (a fixed point) or after 10 rounds.

The last iterate isn't guaranteed to be the best, so the one with the lowest **true** CO₂ (from `evaluate`) is kept. Solution A is the starting incumbent, so B can never be worse than A.

In [5]:
def co2_cost_matrix(payload_leaving, dist, dur, climb):
    """Integer CO2 cost (milligrams) of every arc, given the payload leaving each node."""
    size = len(dist)
    cost = np.zeros((size, size), dtype=np.int64)
    for i in range(size):
        for j in range(size):
            if i != j:
                kg = emissions.co2_kg(dist[i][j], dur[i][j], payload_leaving[i], climb[i][j])
                cost[i][j] = int(round(kg * 1_000_000))
    return cost


# Start from solution A, and keep whichever plan has the lowest TRUE CO2.
payload_estimate = payload_leaving_each_node(routes_distance)
routes_co2, co2_best = routes_distance, co2_A
seen = set()

for iteration in range(1, 11):
    candidate = solve_cvrp(co2_cost_matrix(payload_estimate, distance_m, duration_s, climb_m))
    d, c = evaluate(candidate)

    marker = ""
    if c < co2_best:
        routes_co2, co2_best = candidate, c
        marker = "  <- best so far"
    print(f"  iter {iteration}: {d:6.2f} km   {c:.3f} kg CO2{marker}")

    # Next round uses the payloads this plan actually implies.
    payload_estimate = payload_leaving_each_node(candidate)

    signature = tuple(sorted(tuple(r) for r in candidate))
    if signature in seen:
        print(f"  -> reached a fixed point after {iteration} iterations")
        break
    seen.add(signature)

dist_B, co2_B = evaluate(routes_co2)

print(f"\nSOLUTION B -- minimise CO2")
print(f"  distance: {dist_B:.2f} km")
print(f"  CO2:      {co2_B:.3f} kg\n")
print_routes(routes_co2)

  iter 1:  76.96 km   43.042 kg CO2  <- best so far


  iter 2:  76.96 km   43.042 kg CO2
  -> reached a fixed point after 2 iterations

SOLUTION B -- minimise CO2
  distance: 76.96 km
  CO2:      43.042 kg

  Van 1 (499 kg): Tejgaon Industrial Area -> Rampura -> Badda -> Bashundhara Residential Area -> Uttara Sector 7 -> Mirpur 10 -> Banani -> Gulshan 2 -> Gulshan 1 -> Mohakhali -> Tejgaon Industrial Area
  Van 2 (586 kg): Tejgaon Industrial Area -> Malibagh -> Paltan -> Motijheel -> Shahbagh -> Azimpur -> New Market -> Mohammadpur -> Lalmatia -> Dhanmondi 27 -> Farmgate -> Kawran Bazar -> Tejgaon Industrial Area


### Compare A and B
$$\Delta\text{distance} = 100\left(\frac{D_B}{D_A} - 1\right)\%, \qquad \text{CO}_2\ \text{saved} = 100\left(1 - \frac{C_B}{C_A}\right)\%$$

If B comes out shorter than A, then A wasn't fully optimised and the solver's time limit should be raised.

In [6]:
extra_distance_pct = 100 * (dist_B / dist_A - 1)
co2_saved_pct = 100 * (1 - co2_B / co2_A)

print(f"{'':12} {'distance':>12} {'CO2':>12}")
print(f"{'min-distance':12} {dist_A:9.2f} km {co2_A:9.3f} kg")
print(f"{'min-CO2':12} {dist_B:9.2f} km {co2_B:9.3f} kg")
print(f"{'change':12} {extra_distance_pct:+9.1f} %  {-co2_saved_pct:+9.1f} %")
print()
print(f">>> {extra_distance_pct:+.1f}% distance buys {co2_saved_pct:.1f}% less CO2 <<<")

# A minimised distance, so B can't be shorter -- if it is, A needs more solver time.
if dist_B < dist_A - 0.01:
    print("\n[!] min-CO2 came out shorter than min-distance."
          "\n    Solution A is under-solved. Increase time_limit_s in Cell 4.")

                 distance          CO2
min-distance     76.86 km    43.219 kg
min-CO2          76.96 km    43.042 kg
change            +0.1 %       -0.4 %

>>> +0.1% distance buys 0.4% less CO2 <<<


### How good is the heuristic? Brute-force check
On a small sub-problem (the first stops that fit in one van, up to 7) every visiting order can be tried: $7! = 5040$ routes. The best of them is the true optimum.

The same iterative re-costing is run on this sub-problem and compared:

$$\text{gap} = 100\left(\frac{\text{CO}_2^{\text{heuristic}}}{\text{CO}_2^{\text{optimum}}} - 1\right)\%$$

`np.ix_` picks out the rows and columns of just these nodes to form the sub-matrices.

In [7]:
MAX_BRUTE_STOPS = 7   # 7! = 5,040 orders (instant); 20! would be 2.4e18

# Take stops in index order while they still fit in one van.
subset_stops, subset_load = [], 0.0
for node in range(1, n):
    if len(subset_stops) >= MAX_BRUTE_STOPS:
        break
    if subset_load + demands[node] <= CAPACITY_KG:
        subset_stops.append(node)
        subset_load += demands[node]

idx = np.array([DEPOT] + subset_stops)
m = len(idx)
print(f"Validating on {m-1} stops, {subset_load:.0f} kg of {CAPACITY_KG} kg (one van)")
print()

# Sub-problem matrices, re-indexed 0..m-1 (depot = 0).
sub_dist = distance_m[np.ix_(idx, idx)]
sub_dur = duration_s[np.ix_(idx, idx)]
sub_climb = climb_m[np.ix_(idx, idx)]
sub_dem = demands[idx]


def true_co2_sub(order):
    """Exact CO2 of depot -> order -> depot in the sub-problem."""
    route = [0] + list(order) + [0]
    return sum(emissions.co2_kg(sub_dist[i][j], sub_dur[i][j], payload, sub_climb[i][j])
               for i, j, payload in arcs_with_payload(route, sub_dem))


# --- Brute force: try every order ------------------------------------------
best_order = min(itertools.permutations(range(1, m)), key=true_co2_sub)
best_co2 = true_co2_sub(best_order)


# --- Our heuristic on the same sub-problem: one van, no capacity limit ------
def solve_sub(cost):
    """Best single-van order for an m x m cost matrix (depot excluded from the result)."""
    mgr = pywrapcp.RoutingIndexManager(m, 1, 0)
    rt = pywrapcp.RoutingModel(mgr)
    t = rt.RegisterTransitCallback(
        lambda a, b: int(cost[mgr.IndexToNode(a)][mgr.IndexToNode(b)])
    )
    rt.SetArcCostEvaluatorOfAllVehicles(t)

    p = pywrapcp.DefaultRoutingSearchParameters()
    p.first_solution_strategy = routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC
    p.local_search_metaheuristic = routing_enums_pb2.LocalSearchMetaheuristic.GUIDED_LOCAL_SEARCH
    p.time_limit.FromSeconds(5)

    sol = rt.SolveWithParameters(p)
    i, route = rt.Start(0), []
    while not rt.IsEnd(i):
        route.append(mgr.IndexToNode(i))
        i = sol.Value(rt.NextVar(i))
    return route[1:]


payload_est = [sub_dem[1:].sum()] * m
for _ in range(10):
    order = solve_sub(co2_cost_matrix(payload_est, sub_dist, sub_dur, sub_climb))
    for i, j, payload in arcs_with_payload([0] + order + [0], sub_dem):
        payload_est[i] = payload

heuristic_co2 = true_co2_sub(order)
gap = 100 * (heuristic_co2 / best_co2 - 1)

print(f"Brute-force optimum : {best_co2:.4f} kg   order {best_order}")
print(f"Our heuristic       : {heuristic_co2:.4f} kg   order {tuple(order)}")
print(f"OPTIMALITY GAP      : {gap:.2f}%")
print()
print("Put this number in your README. A measured gap is a result;")
print("an unmeasured one is a hole reviewers will find.")

Validating on 7 stops, 342 kg of 600 kg (one van)



Brute-force optimum : 10.7570 kg   order (3, 2, 1, 4, 7, 6, 5)
Our heuristic       : 10.7570 kg   order (3, 2, 1, 4, 7, 6, 5)
OPTIMALITY GAP      : 0.00%

Put this number in your README. A measured gap is a result;
an unmeasured one is a hole reviewers will find.


### Save
Routes for both objectives → `routes_{city}.csv`; headline comparison including the optimality gap → `comparison_{city}.csv`.

In [8]:
rows = []
for label, routes in [("min_distance", routes_distance), ("min_co2", routes_co2)]:
    for v, route in enumerate(routes, 1):
        rows.append({
            "objective": label,
            "vehicle": v,
            "route": " ".join(map(str, route)),
            "load_kg": sum(demands[i] for i in route),
        })
pd.DataFrame(rows).to_csv(f"{DATA_DIR}/routes_{CITY}.csv", index=False)

pd.DataFrame([{
    "city": CITY,
    "distance_km_min_distance": round(dist_A, 3),
    "co2_kg_min_distance": round(co2_A, 4),
    "distance_km_min_co2": round(dist_B, 3),
    "co2_kg_min_co2": round(co2_B, 4),
    "extra_distance_pct": round(extra_distance_pct, 2),
    "co2_saved_pct": round(co2_saved_pct, 2),
    "heuristic_gap_pct": round(gap, 2),
}]).to_csv(f"{DATA_DIR}/comparison_{CITY}.csv", index=False)

print(f"Saved routes_{CITY}.csv and comparison_{CITY}.csv")

Saved routes_dhaka.csv and comparison_dhaka.csv


## Conclusions

| | Dhaka | Sylhet |
|---|---|---|
| Min-distance plan | 76.86 km · 43.219 kg CO₂ | 62.57 km · 26.619 kg CO₂ |
| Min-CO₂ plan | 76.96 km · 43.042 kg CO₂ | 62.57 km · 26.619 kg CO₂ |
| Trade-off | +0.13% distance → **−0.41% CO₂** | **no change, same plan** |
| Optimality gap (brute force) | 0.00% | 0.00% |

- **Dhaka:** the CO₂-optimal plan differs from the distance-optimal one by a single move, and no stop changes van. Van 2 visits Dhanmondi 27 → Lalmatia → Mohammadpur in reverse order, so it drops the heaviest of the three parcels (Mohammadpur, 54 kg) first instead of last. This is the load effect at work, and it is worth **177 g of CO₂** on a 77 km round.
- **Sylhet:** the distance-optimal plan is already the CO₂-optimal one. The re-costing loop oscillated (iterations 1 and 3 were *worse*, at 27.187 kg), which is why the loop keeps the best plan by true CO₂ rather than the last one.
- **The heuristic is accurate:** on the 7-stop sub-problem it matches the brute-force optimum exactly in both cities, so the small difference is not a solver failure.
- **Takeaway:** on these networks, minimising distance already gives (almost exactly) the minimum-carbon routes.